In [ ]:
import pandas as p 
import numpy as np 
import matplotlib.pyplot  as plt 
import seaborn as sns 

In [ ]:
df=p.read_csv('ipl_2026_deliveries.csv')


In [ ]:
df.info()

In [ ]:
df.isna().sum()

Total No. Of matches

In [ ]:
len(df['match_id'].unique())

#How many teams are playing 

In [ ]:
df['batting_team'].unique()

Feature Engineering

In [ ]:
df['total_runs']=df['runs_of_bat'] + df['extras']


In [ ]:
df["total_6s"]=df['runs_of_bat']==6
df['total_6s']=df['total_6s'].astype(int)
df["total_4s"]=df['runs_of_bat']==4
df['total_4s']=df['total_4s'].astype(int)

In [ ]:
df['is_wicket']=0

df.loc[df['wicket_type'].notnull(),'is_wicket']=1


# Over Phase 

In [ ]:
def getph(o):
    if o<6.0:
        return "PowerPlay"
    elif o<16: 
        return "Middle"
    else:
        return "Death"

df['Over_Phase']=df['over'].apply(getph)


## Batting Analysis

# Which Team Has Scored the most runs?

In [ ]:
b=df.groupby('batting_team',as_index=False)['total_runs'].sum()
ma=b.sort_values(by='total_runs',ascending=False)
ma

In [ ]:
plt.figure(figsize=(9,8))
plt.bar(ma['batting_team'],ma['total_runs'],color=['#EA1A85','#EC1C24','#3A225D','#F7A721','#1C1C1C','#F9CD05','#0078BC','#004BA0','#A0E6FF','#ED1B24'],edgecolor='black')

plt.xlabel('Teams')
plt.ylabel('Runs')
plt.title('Runs Scored Graph')

for i,v in zip(b['batting_team'],b['total_runs']):
    plt.text(x=i,y=v,s=v,va='bottom')

plt.gca().set_axisbelow(True)
plt.grid()  

# Which Team has Hits the most 6's and 4's

In [ ]:
total_6s=df.groupby('batting_team',as_index=False).agg(
    six=('total_6s','sum')
)
total_6s=total_6s.sort_values(by='six',ascending=False)
total_4s=df.groupby('batting_team',as_index=False).agg(
    four=('total_4s','sum')
)
total_4s=total_4s.sort_values(by='four',ascending=False)
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

# Sixes
ax[0].bar(
    total_6s['batting_team'],
    total_6s['six'],
    color=['#EA1A85','#EC1C24','#3A225D','#F7A721','#1C1C1C',
           '#F9CD05','#0078BC','#004BA0','#A0E6FF','#ED1B24'],
    edgecolor='black'
)

ax[0].set_xlabel('Teams')
ax[0].set_ylabel("6's")
ax[0].set_title('Total Sixes By Each Team')
ax[0].grid(axis='y')
ax[0].set_axisbelow(True)

for i, v in zip(total_6s['batting_team'], total_6s['six']):
    ax[0].text(i, v, v, ha='center', va='bottom')


#fours
ax[1].bar(
    total_4s['batting_team'],
    total_4s['four'],
    color=['#EA1A85','#EC1C24','#3A225D','#F7A721','#1C1C1C',
           '#F9CD05','#0078BC','#004BA0','#A0E6FF','#ED1B24'],
    edgecolor='black'
)

ax[1].set_xlabel('Teams')
ax[1].set_ylabel("4's")
ax[1].set_title('Total Fours By Each Team')
ax[1].grid(axis='y')
ax[1].set_axisbelow(True)

for i, v in zip(total_4s['batting_team'], total_4s['four']):
    ax[1].text(i, v, v, ha='center', va='bottom')

plt.tight_layout()
plt.suptitle("Teams with 6's and 4's")
ax[0].tick_params(axis='x', rotation=40)
ax[1].tick_params(axis='x', rotation=40)
plt.show()

In [ ]:
# RR Hits more no. of 6's and 4's

# LSG hits least no. of 6's and PBKS hits least no. of 4's

# Which team has hit the most number of boundaries

In [ ]:
df['boundaries']=df['total_6s']+df['total_4s']

In [ ]:
plt.figure(figsize=(9,7))
bound=df.groupby('batting_team',as_index=False)[['boundaries','total_4s','total_6s']].sum().sort_values(by='boundaries',ascending=False)
sns.barplot(
    x=bound['batting_team'],
    y=bound['boundaries'],
    
    palette=['#EA1A85','#EC1C24','#3A225D','#F7A721','#1C1C1C',
             '#F9CD05','#0078BC','#004BA0','#A0E6FF','#ED1B24'],
    edgecolor='black'
)

for i,v in zip(bound['batting_team'],bound['boundaries']): 
    plt.text(x=i,y=v,s=v,ha='center',va='bottom')


In [ ]:
# RR Hits More boundaries whereas PBKS Hits less

# Now combining all three boundaries vs 4's vs 6's

In [ ]:
m = bound.melt(
    id_vars='batting_team',
    value_vars=['boundaries','total_4s','total_6s'],
    var_name='Type',
    value_name='Count'
)

plt.figure(figsize=(12,6))

a=sns.barplot(
    data=m,
    x='batting_team',
    y='Count',
    hue='Type',
    edgecolor='black'
)

plt.title('Boundaries vs 4s vs 6s by Team')
plt.xticks(rotation=45)

for container in a.containers:
    a.bar_label(container)

plt.show()


# How many runs did each team score in powerplay,middle,death

In [ ]:
r=df.groupby(['batting_team','Over_Phase'],as_index=False)['total_runs'].sum()
plt.figure(figsize=(12,8))
an=sns.barplot(x='batting_team',y='total_runs',data=r,hue='Over_Phase',edgecolor='black',palette=['b','y','r'],hue_order=['PowerPlay','Middle','Death'])
plt.title('Runs Scored In Different Phases')
plt.legend(loc='upper left')
for x in an.containers:
    an.bar_label(x,padding=2)

In [ ]:
In PowerPlay :- RR Scored Most

In Middle :-  RCB scored most

In Death :- SRH scored most

# Batsman analysis

#Top 10 Scorers in ipl 2026 

In [ ]:
b=df.groupby('striker',as_index=False)['total_runs'].sum().sort_values(by='total_runs',ascending=False).head(10)
plt.figure(figsize=(12,6))
an=sns.barplot(x='total_runs',y='striker',data=b,edgecolor='black',label="Runs",width=0.7,color='orange')
plt.xticks(rotation=0)
plt.grid(axis='y')
plt.title('Top 10 Scorers')
plt.gca().set_axisbelow(True)
for x in an.containers:
    an.bar_label(x,padding=2)

Vaibhav Suryavansi is the current orange cap holder

# Batsman scored most boundaries

# most 6's

In [ ]:
s=df.groupby('striker',as_index=False)['total_6s'].sum().sort_values(by='total_6s',ascending=False).head(10)
f=df.groupby('striker',as_index=False)['total_4s'].sum().sort_values(by='total_4s',ascending=False).head(10)

fig, ax = plt.subplots(1,2,figsize=(20,10))

bars1 = ax[0].barh(
    s['striker'],
    s['total_6s'],
    edgecolor='black'
)

ax[0].set_title("Top 10 Batters by Sixes", fontsize=14)
ax[0].set_xlabel("Total Sixes")
ax[0].set_ylabel("Striker")
ax[0].grid(axis='x', linestyle='--', alpha=0.7)
ax[0].set_axisbelow(True)

# Add labels
ax[0].bar_label(bars1, padding=3)



# ------------------ Fours ------------------
bars2 = ax[1].barh(
    f['striker'],
    f['total_4s'],
    edgecolor='black'
)

ax[1].set_title("Top 10 Batters by Fours", fontsize=14)
ax[1].set_xlabel("Total Fours")
ax[1].set_ylabel("Striker")
ax[1].grid(axis='x', linestyle='--', alpha=0.7)
ax[1].set_axisbelow(True)

# Add labels
ax[1].bar_label(bars2, padding=3)
plt.suptitle('Most Number Of Boundaries')

plt.tight_layout()
plt.show()

Vaibhav Hits most no. of 6's(18) and jaiswal hits more no. of 4's (19)

# Strike rate of top batsman

In [ ]:

# Balls faced (wides excluded)
df['Ball_Faced'] = (df['wide'] == 0).astype(int)

# Calculate Runs and Balls
sr = df.groupby('striker', as_index=False).agg(
    Runs=('total_runs', 'sum'),
    Balls=('Ball_Faced', 'sum')
)

# Strike Rate
sr['Strike_Rate'] = ((sr['Runs'] / sr['Balls']) * 100).round(2)

# Minimum 50 balls faced
sr = sr[sr['Balls'] >= 50]

# Top 10 Strike Rates
sr = sr.sort_values(
    by='Strike_Rate',
    ascending=False
).head(10)

# Plot
plt.figure(figsize=(12, 10))

an = sns.barplot(
    x='striker',
    y='Strike_Rate',
    data=sr,
    edgecolor='black',
    color='yellow',
    width=0.5
)

plt.xticks(rotation=40)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.gca().set_axisbelow(True)

plt.title('Top 10 Strike Rates (Min 50 Balls Faced)')
plt.xlabel('Batters')
plt.ylabel('Strike Rate')

for i in an.containers: 
    an.bar_label(i)
plt.tight_layout()
plt.show()

Vaibhav has the highest strike rate so far

# Bowler Analysis 

In [ ]:
df['bowler_wkt'] = (
    (df['is_wicket'] == 1) &
    (df['wicket_type'] != 'runout')
).astype(int)
wk=df.groupby('bowler',as_index=False).agg(
    Total_wk=('bowler_wkt','sum')
)
wk=wk.sort_values(by='Total_wk',ascending=False)
wk=wk.head(5)

an=sns.barplot(x=wk['bowler'],y=wk['Total_wk'],color='purple',label='wickets')
plt.legend()
plt.xticks(rotation=40)
plt.gca().set_axisbelow(True)
plt.grid(axis='x',linestyle='-',alpha=0.8)
plt.title('Top 10 Bowlers Of IPL 2026')
for i in an.containers: 
    an.bar_label(i)

Ravi bishnoi is the leading wicket taker so far

#  Which Bowler Has Best Economy Rate # Most Green Dot Bowls Bowled

#Economy Rate= Runs Conceded / Overs Bowled

In [ ]:
df['Legal_Ball'] = (
    (df['wide'] == 0) &
    (df['noballs'] == 0)
).astype(int) 

eco = df.groupby('bowler', as_index=False).agg(
    Runs_Conceded=('total_runs', 'sum'),
    Balls=('Legal_Ball', 'sum')
)

eco['Overs'] = eco['Balls'] / 6
eco['Economy'] = (
    eco['Runs_Conceded'] / eco['Overs']
).round(2)
  
eco = eco[eco['Balls'] >= 60]   

eco = eco.sort_values(
    by='Economy',
    ascending=True
).head(10)


In [ ]:
df['Dot_Ball'] = (
    (df['total_runs'] == 0) &
    (df['wide'] == 0) &
    (df['noballs'] == 0)
).astype(int) 

dot = df.groupby('bowler', as_index=False).agg(
    Dot_Balls=('Dot_Ball', 'sum')
)

dot = dot.sort_values(
    by='Dot_Balls',
    ascending=False
).head(10) 

fig, ax = plt.subplots(1, 2, figsize=(18, 8))

# ---------------- Economy ----------------
eco_plot = sns.barplot(
    x='bowler',
    y='Economy',
    data=eco,
    edgecolor='black',
    color='violet',
    ax=ax[0]
)

ax[0].set_title('Top 10 Most Economical Bowlers\n(Min 10 Overs Bowled)')
ax[0].set_xlabel('Bowler')
ax[0].set_ylabel('Economy')
ax[0].tick_params(axis='x', rotation=40)
ax[0].grid(axis='y', linestyle='--', alpha=0.7)
ax[0].set_axisbelow(True)

for container in eco_plot.containers:
    eco_plot.bar_label(container, fmt='%.2f')

# ---------------- Dot Balls ----------------
dot_plot = sns.barplot(
    x='bowler',
    y='Dot_Balls',
    data=dot,
    edgecolor='black',
    color='g',
    ax=ax[1]
)

ax[1].set_title('Top 10 Bowlers by Dot Balls')
ax[1].set_xlabel('Bowler')
ax[1].set_ylabel('Dot Balls')
ax[1].tick_params(axis='x', rotation=40)
ax[1].grid(axis='y', linestyle='--', alpha=0.7)
ax[1].set_axisbelow(True)

for container in dot_plot.containers:
    dot_plot.bar_label(container)

plt.tight_layout()
plt.show()

M Sami is the best economical bowler so far having 5.33 rate and also delivered most green dot balls 38

# How are Batsman getting out 

In [ ]:
o=df.groupby('wicket_type').agg(
    Count=('wicket_type','count')
).reset_index()

o = o.sort_values('Count', ascending=False)

plt.figure(figsize=(10,6))

an=sns.countplot(
    data=df,
    y='wicket_type',
    hue='wicket_type',
    palette='viridis',
    legend=False,
    edgecolor='black'
)

plt.title('Wicket Types Distribution', fontsize=16, fontweight='bold')
plt.xlabel('Count')
plt.ylabel('Wicket Type')

for i in an.containers:
    an.bar_label(i)
plt.tight_layout()
plt.show()

Mostly Batsman getting out by caughts

# Over by Over Runs

In [ ]:
df['over_no']=df['over'].astype(int)

on=df.groupby('over_no').agg(
    Runs=('total_runs','sum')
)
on=on.reset_index()

plt.figure(figsize=(12,6))

ax = sns.lineplot(
    data=on,
    x='over_no',
    y='Runs',
    marker='o',
    linewidth=3
)

ax.fill_between(
    on['over_no'],
    on['Runs'],
    alpha=0.25
)
for x, y in zip(on['over_no'], on['Runs']):
    plt.text(
        x, y +3,      # thoda upar
        str(y),
        ha='center',
        va='bottom',
        fontsize=9,
        fontweight='bold'
    )
plt.title('Runs Distribution Across Overs', fontsize=16, fontweight='bold')
plt.xlabel('Over')
plt.ylabel('Runs')
plt.xticks(range(0, 21))
plt.grid(alpha=0.3)

plt.show()

In Powerplay most runs were scored so far where as death over has least scores

# Which Team is better in PowerPlay , Middle , Death

In [ ]:
be=df.groupby(['batting_team','Over_Phase'],as_index=False)['total_runs'].sum()
be=df.pivot_table(values='total_runs',columns='Over_Phase',index='batting_team',aggfunc='sum').reset_index()
be
plt.figure(figsize=(10,6))

sns.heatmap(
    be.set_index('batting_team'),
    annot=True,
    fmt='.0f',
    cmap='rainbow'
)

plt.title('Team-wise Runs Across Over Phases')
plt.show()